# Research Agent RL — Week 3: Multi-Turn GRPO

**Setup:** GPU T4 x1, Internet ON

This notebook trains the LLM with **multi-turn GRPO** — the same core idea
as DeepSeek-R1, but with **real tool execution** at each step:

```
LLM generates action → real HotpotQA search index executes → real result fed back → repeat
```

This closes the main gap vs production agents (like OpenAI Deep Research)
where the LLM sees actual retrieval results during training, not hallucinated ones.

**GRPO mechanics:**
1. For each question, run G=4 independent episodes with real tool calls
2. Score each episode: correctness + format + efficiency
3. Normalise rewards within the group → advantages (no critic network needed)
4. One forward pass over the full trajectory extracts log-probs of LLM-generated tokens only
5. Policy gradient update: push LLM toward higher-reward trajectories

## 1. Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — enable T4 in Settings!')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q transformers peft accelerate datasets pyyaml tqdm bitsandbytes

## 3. Load SFT checkpoint (Week 1 adapter)

We start from the SFT model (already knows the JSON trace format) and
further improve it with GRPO.

In [ ]:
import os, shutil, glob

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

if not os.path.exists(ADAPTER_DIR):
    print('Downloading Week 1 checkpoint...')
    !mkdir -p /tmp/w1_out
    !kaggle kernels output wuyue22/rl-sft-research -p /tmp/w1_out
    src = '/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/final'
    if os.path.exists(src):
        os.makedirs('checkpoints/qwen-sft', exist_ok=True)
        shutil.copytree(src, ADAPTER_DIR)
    else:
        ckpts = sorted(glob.glob('/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/checkpoint-*'))
        if ckpts:
            shutil.copytree(ckpts[-1], ADAPTER_DIR)
        else:
            print('[ERROR] No checkpoint found.')
else:
    print(f'Checkpoint already present at {ADAPTER_DIR}')

!ls -lh {ADAPTER_DIR}

In [ ]:
import json, torch
from peft import PeftModel, get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_DIR = 'checkpoints/qwen-sft/final'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    base_model_name = json.load(f).get('base_model_name_or_path', 'Qwen/Qwen2.5-0.5B-Instruct')

print(f'Loading base model: {base_model_name}')
base = AutoModelForCausalLM.from_pretrained(
    base_model_name, device_map={'': 0}, torch_dtype=torch.float16, trust_remote_code=True
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.config.use_cache = False
model.enable_input_require_grads()   # needed for gradient flow through LoRA

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 4. Load HotpotQA & build real search index

In [ ]:
from datasets import load_dataset
from agent.tools import ToolExecutor

N_TRAIN = 500
N_VAL   = 100

print('Loading HotpotQA...')
hf       = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
train_hf = hf['train'].shuffle(seed=42)
val_hf   = hf['validation']

train_questions = [
    {'question': train_hf[i]['question'], 'answer': train_hf[i]['answer']}
    for i in range(N_TRAIN)
]
val_questions = [
    {'question': val_hf[i]['question'], 'answer': val_hf[i]['answer']}
    for i in range(N_VAL)
]

# Build real search index from val contexts
tool_executor = ToolExecutor(top_k=2)
tool_executor.build_from_hotpotqa(val_hf, index_path='data/tool_index.jsonl')
print(f'Tool index: {len(tool_executor)} paragraphs')
print(f'Train questions: {len(train_questions)} | Val: {len(val_questions)}')

## 5. Smoke-test: one interactive episode

In [ ]:
from rl.multi_turn_grpo import collect_episode
from data.sft_dataset import SYSTEM_PROMPT

model.eval()
q = val_questions[0]
ep = collect_episode(
    q['question'], q['answer'],
    model, tokenizer, tool_executor,
    system_prompt=SYSTEM_PROMPT,
    max_steps=4, temperature=0.1, device=DEVICE,
)

print(f'Q      : {ep.question}')
print(f'Gold   : {ep.gold_answer}')
print(f'Correct: {ep.correct}  |  Steps: {ep.n_steps}')
print(f'\n--- Full trajectory ---')
print(ep.full_text[-800:])   # show end of trajectory

## 6. Multi-turn GRPO training

In [ ]:
from rl.multi_turn_grpo import train as mt_grpo_train
import torch

torch.cuda.empty_cache()

# Hyperparameters — tuned for T4 16GB
# batch_size=2 questions × G=4 episodes = 8 episodes per update
# Each episode: up to 5 steps × ~256 tokens = ~1280 tokens
# One forward pass per episode for log-probs → manageable VRAM

log = mt_grpo_train(
    model=model,
    tokenizer=tokenizer,
    tool_executor=tool_executor,
    train_questions=train_questions,
    val_questions=val_questions,
    system_prompt=SYSTEM_PROMPT,
    output_dir='checkpoints/mt-grpo',
    n_epochs=30,
    batch_size=2,     # questions per update
    G=4,              # episodes per question (GRPO group size)
    lr=5e-6,
    max_steps=5,
    alpha=0.1,        # step penalty weight
    beta=0.05,        # JSD penalty weight
    epsilon=0.05,     # JSD threshold
    val_every=5,
    save_every=10,
    device=DEVICE,
)

print('Training complete!')

## 7. Evaluate trained model

In [ ]:
from rl.multi_turn_grpo import collect_episode
from tqdm import tqdm

model.eval()
N_EVAL = 100

correct, total_steps = 0, 0
results = []

for q in tqdm(val_questions[:N_EVAL], desc='Evaluating'):
    ep = collect_episode(
        q['question'], q['answer'],
        model, tokenizer, tool_executor,
        system_prompt=SYSTEM_PROMPT,
        max_steps=5, temperature=0.1, device=DEVICE,
    )
    correct     += int(ep.correct)
    total_steps += ep.n_steps
    results.append({
        'question': ep.question, 'gold': ep.gold_answer,
        'correct': ep.correct, 'n_steps': ep.n_steps,
    })

acc       = correct / N_EVAL
avg_steps = total_steps / N_EVAL
print(f'\nMulti-turn GRPO model')
print(f'  Accuracy : {acc:.3f}')
print(f'  Avg steps: {avg_steps:.1f}')
print(f'  Efficiency (acc/steps): {acc/max(avg_steps,1):.4f}')

## 8. Save everything

In [ ]:
import json, shutil

# Copy final adapter to output
shutil.copytree('checkpoints/mt-grpo/final', '/kaggle/working/mt_grpo_adapter', dirs_exist_ok=True)
print('Adapter saved → /kaggle/working/mt_grpo_adapter')

# Eval results
with open('/kaggle/working/mt_grpo_eval_results.json', 'w') as f:
    json.dump({
        'summary': {'accuracy': acc, 'avg_steps': avg_steps,
                    'efficiency': acc / max(avg_steps, 1), 'n_eval': N_EVAL},
        'results': results,
    }, f, indent=2)
print('Eval results saved → /kaggle/working/mt_grpo_eval_results.json')

# Training log
with open('/kaggle/working/mt_grpo_training_log.json', 'w') as f:
    json.dump(log, f, indent=2)
print('Training log saved → /kaggle/working/mt_grpo_training_log.json')

print('\nAll outputs in Kaggle Output tab.')